# WavLM + Prosody Training Pipeline v2
## Combined Features for Better Discrimination

**Features:**
- WavLM (768d) - learned acoustic representations
- Prosody (21d) - F0, energy, MFCCs, pause patterns

**Evaluation:** Comedian-level holdout

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
print('Drive mounted')

In [ ]:
!pip install -q transformers datasets torch torchaudio librosa scikit-learn

import os, json, torch, numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score
from tqdm import tqdm
import librosa

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SR = 16000
print(f'Device: {DEVICE}')

In [ ]:
# Load utterances
!wget -q -O /tmp/utt.jsonl.gz https://raw.githubusercontent.com/Das-rebel/chuck-audio-notebooks/main/utterances_clean.jsonl.gz
!gunzip -f /tmp/utt.jsonl.gz

utterances = [json.loads(l) for l in open('/tmp/utt.jsonl')]
print(f'Utterances: {len(utterances)}')

In [ ]:
# Load WavLM embeddings
WAVLM_DIR = '/content/gdrive/MyDrive/wavlm_utterance_safe'

wavlm_data = {}
for fname in os.listdir(WAVLM_DIR):
    if fname.endswith('.json') and fname != 'checkpoint.json':
        with open(os.path.join(WAVLM_DIR, fname)) as f:
            data = json.load(f)
            vid = data['video_id']
            wavlm_data[vid] = {}
            for emb in data['embeddings']:
                # Round start to 2 decimal places for matching
                start_key = round(emb['start'], 2)
                wavlm_data[vid][start_key] = emb['embedding']

print(f'WavLM videos: {len(wavlm_data)}')

In [ ]:
# Prosody extraction function
def extract_prosody(audio_path, start, end):
    """Extract 21-dim prosody features from audio segment."""
    try:
        # Load audio segment
        y, sr = librosa.load(audio_path, offset=start, duration=end-start, sr=SR)
        
        if len(y) < SR * 0.05:  # Too short
            return [0] * 21
        
        features = []
        
        # 1. RMS energy (1 dim)
        rms = librosa.feature.rms(y=y)[0]
        features.extend([rms.mean(), rms.std()])
        
        # 2. Zero crossing rate (1 dim)
        zcr = librosa.feature.zero_crossing_rate(y)[0]
        features.extend([zcr.mean(), zcr.std()])
        
        # 3. F0/pitch (2 dims)
        try:
            pitches, magnitudes = librosa.piptrack(y=y, sr=sr)
            # Get pitch values where magnitude > threshold
            pitch_vals = pitches[pitches > 0]
            if len(pitch_vals) > 0:
                features.extend([pitch_vals.mean(), pitch_vals.std()])
            else:
                features.extend([0, 0])
        except:
            features.extend([0, 0])
        
        # 4. MFCCs (13 dims)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        for i in range(13):
            features.extend([mfccs[i].mean(), mfccs[i].std()])
        
        # 5. Spectral features (2 dims)
        spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
        features.extend([spec_cent.mean(), spec_cent.std()])
        
        return features[:21]  # Ensure 21 dims
        
    except Exception as e:
        return [0] * 21

In [ ]:
# Find audio files
audio_files = {}
for root, dirs, files in os.walk('/content/gdrive/MyDrive/chuckle_audio_all'):
    for f in files:
        if f.endswith(('.mp3', '.wav')):
            vid = os.path.splitext(f)[0]
            if vid not in audio_files:
                audio_files[vid] = os.path.join(root, f)
print(f'Audio files: {len(audio_files)}')

In [ ]:
# Match utterances with WavLM + prepare for prosody extraction
# First match with WavLM
matched = []
for u in utterances:
    vid = u['video_id']
    if vid not in wavlm_data:
        continue
    start_key = round(u['start'], 2)
    if start_key not in wavlm_data[vid]:
        continue
    
    # Skip very short or very long utterances
    duration = u['end'] - u['start']
    if duration < 0.05 or duration > 30:
        continue
    
    matched.append({
        'video_id': vid,
        'start': u['start'],
        'end': u['end'],
        'text': u.get('text', ''),
        'label': u['label'],
        'wavlm': wavlm_data[vid][start_key]
    })

print(f'Matched with WavLM: {len(matched)}')
pos = sum(1 for m in matched if m['label'] == 1)
print(f'Positive: {pos} ({100*pos/len(matched):.2f}%)')

In [ ]:
# Extract prosody for matched utterances (sample for speed)
# For full training, this takes ~30 min on GPU
print('Extracting prosody features...')
print('This will take ~20-30 minutes for all data')

for m in tqdm(matched):
    vid = m['video_id']
    if vid in audio_files:
        prosody = extract_prosody(audio_files[vid], m['start'], m['end'])
    else:
        prosody = [0] * 21
    m['prosody'] = prosody

print('Prosody extraction complete!')

In [ ]:
# Save extracted features to Drive for reuse
FEATURES_DIR = '/content/gdrive/MyDrive/wavlm_prosody_features'
os.makedirs(FEATURES_DIR, exist_ok=True)

with open(os.path.join(FEATURES_DIR, 'matched_features.json'), 'w') as f:
    json.dump(matched, f)

print(f'Features saved to {FEATURES_DIR}')

In [ ]:
# Comedian-level holdout split
video_groups = {}
for m in matched:
    vid = m['video_id']
    if vid not in video_groups:
        video_groups[vid] = []
    video_groups[vid].append(m)

videos = list(video_groups.keys())
np.random.seed(42)
np.random.shuffle(videos)

# 10% test, 10% val, 80% train
n = len(videos)
test_vids = set(videos[:n//10])
val_vids = set(videos[n//10:n//5])
train_vids = set(videos[n//5:])

train_data = [m for v in train_vids for m in video_groups[v]]
val_data = [m for v in val_vids for m in video_groups[v]]
test_data = [m for v in test_vids for m in video_groups[v]]

print(f'Train: {len(train_data)} ({len(train_vids)} videos)')
print(f'Val: {len(val_data)} ({len(val_vids)} videos)')
print(f'Test: {len(test_data)} ({len(test_vids)} videos)')

train_pos = sum(1 for m in train_data if m['label'] == 1)
val_pos = sum(1 for m in val_data if m['label'] == 1)
test_pos = sum(1 for m in test_data if m['label'] == 1)
print(f'\nTrain pos: {train_pos} ({100*train_pos/len(train_data):.2f}%)')
print(f'Val pos: {val_pos} ({100*val_pos/len(val_data):.2f}%)')
print(f'Test pos: {test_pos} ({100*test_pos/len(test_data):.2f}%)')

In [ ]:
# Prepare feature arrays
def prepare(data):
    X_wavlm = np.array([m['wavlm'] for m in data], dtype=np.float32)
    X_prosody = np.array([m['prosody'] for m in data], dtype=np.float32)
    y = np.array([m['label'] for m in data], dtype=np.float32)
    return X_wavlm, X_prosody, y

X_train_w, X_train_p, y_train = prepare(train_data)
X_val_w, X_val_p, y_val = prepare(val_data)
X_test_w, X_test_p, y_test = prepare(test_data)

print(f'WavLM: train={X_train_w.shape}, val={X_val_w.shape}, test={X_test_w.shape}')
print(f'Prosody: train={X_train_p.shape}, val={X_val_p.shape}, test={X_test_p.shape}')

In [ ]:
# Fusion MLP Model
class FusionClassifier(torch.nn.Module):
    def __init__(self, wavlm_dim=768, prosody_dim=21):
        super().__init__()
        # Prosody projection
        self.prosody_net = torch.nn.Sequential(
            torch.nn.Linear(prosody_dim, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 32)
        )
        
        # WavLM + Prosody fusion
        self.fusion_net = torch.nn.Sequential(
            torch.nn.Linear(wavlm_dim + 32, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(256, 64),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.2),
            torch.nn.Linear(64, 1),
            torch.nn.Sigmoid()
        )
    
    def forward(self, wavlm, prosody):
        prosody_emb = self.prosody_net(prosody)
        combined = torch.cat([wavlm, prosody_emb], dim=1)
        return self.fusion_net(combined).squeeze()

model = FusionClassifier().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Class weight for imbalance
pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()]).to(DEVICE)
print(f'Pos weight: {pos_weight.item():.2f}')

In [ ]:
# Training
BATCH = 256
EPOCHS = 30

best_val_f1 = 0
best_state = None

for epoch in range(EPOCHS):
    model.train()
    indices = np.random.permutation(len(X_train_w))
    total_loss = 0
    
    for i in range(0, len(indices), BATCH):
        idx = indices[i:i+BATCH]
        X_w = torch.tensor(X_train_w[idx]).to(DEVICE)
        X_p = torch.tensor(X_train_p[idx]).to(DEVICE)
        y_b = torch.tensor(y_train[idx]).to(DEVICE)
        
        optimizer.zero_grad()
        preds = model(X_w, X_p)
        
        # Weighted BCE
        weights = torch.where(y_b == 1, pos_weight, torch.ones_like(y_b))
        loss = (weights * torch.nn.functional.binary_cross_entropy(preds, y_b, reduction='none')).mean()
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    # Evaluate
    model.eval()
    with torch.no_grad():
        val_preds = model(torch.tensor(X_val_w).to(DEVICE), torch.tensor(X_val_p).to(DEVICE)).cpu().numpy()
        val_preds_bin = (val_preds > 0.5).astype(int)
        val_f1 = f1_score(y_val, val_preds_bin, zero_division=0)
        
        test_preds = model(torch.tensor(X_test_w).to(DEVICE), torch.tensor(X_test_p).to(DEVICE)).cpu().numpy()
        test_preds_bin = (test_preds > 0.5).astype(int)
        test_f1 = f1_score(y_test, test_preds_bin, zero_division=0)
        test_p = precision_score(y_test, test_preds_bin, zero_division=0)
        test_r = recall_score(y_test, test_preds_bin, zero_division=0)
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = model.state_dict().copy()
    
    print(f'Epoch {epoch+1:2d} | Loss: {total_loss:.4f} | Val F1: {val_f1:.4f} | Test F1: {test_f1:.4f} P: {test_p:.3f} R: {test_r:.3f}')
    

In [ ]:
# Final evaluation
model.load_state_dict(best_state)
model.eval()

with torch.no_grad():
    test_preds = model(torch.tensor(X_test_w).to(DEVICE), torch.tensor(X_test_p).to(DEVICE)).cpu().numpy()
    test_preds_bin = (test_preds > 0.5).astype(int)
    
    final_f1 = f1_score(y_test, test_preds_bin, zero_division=0)
    final_p = precision_score(y_test, test_preds_bin, zero_division=0)
    final_r = recall_score(y_test, test_preds_bin, zero_division=0)
    final_acc = (test_preds_bin == y_test).mean()

print('\n=== FINAL RESULTS ===')
print(f'Val F1: {best_val_f1:.4f}')
print(f'Test F1: {final_f1:.4f}')
print(f'Test Precision: {final_p:.4f}')
print(f'Test Recall: {final_r:.4f}')
print(f'Test Accuracy: {final_acc:.4f}')

In [ ]:
# Save
OUTPUT_DIR = '/content/gdrive/MyDrive/wavlm_prosody_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

torch.save({'model_state_dict': best_state}, os.path.join(OUTPUT_DIR, 'fusion_model.pt'))

results = {
    'val_f1': best_val_f1,
    'test_f1': final_f1,
    'test_precision': final_p,
    'test_recall': final_r,
    'test_accuracy': final_acc,
    'n_train': len(train_data),
    'n_val': len(val_data),
    'n_test': len(test_data),
}

with open(os.path.join(OUTPUT_DIR, 'results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print(f'\nSaved to {OUTPUT_DIR}')